# 01 - Data Exploration
**Goal:** understand the raw Netflix dataset before cleaning it: its size, columns, missing values, and oddities.

In [3]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)


In [16]:
df = pd.read_csv("C:\\Users\\sreej\\codetech_projects\\NetflixContentVisualization\\netflix_content_visualization\\data\\raw\\netflix_titles.csv")
print("Shape:", df.shape)
df.head()

Shape: (8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirst..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, ...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets o..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari,...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adve...","To protect his family from a powerful drug lord, skilled..."
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down among the inc..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam Khan, Ahsaa...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV Comedies",In a city of coaching centers known to train India’s fin...


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


## Missing values

In [9]:
missing = pd.DataFrame({
    "missing": df.isnull().sum(),
    "percent": (df.isnull().mean() * 100).round(2),
}).sort_values("missing", ascending=False)
missing

,missing,percent
director,2634,29.91
country,831,9.44
cast,825,9.37
date_added,10,0.11
rating,4,0.05
duration,3,0.03
show_id,0,0.00
type,0,0.00
title,0,0.00
release_year,0,0.00


## Duplicates

In [10]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate show_ids:", df["show_id"].duplicated().sum())

Duplicate rows: 0
Duplicate show_ids: 0


## Categorical columns

In [12]:
df["type"].value_counts()

type
Movie      6131
TV Show    2676
Name: count, dtype: int64

In [13]:
df["rating"].value_counts(dropna=False)

rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NaN            4
NC-17          3
UR             3
66 min         1
74 min         1
84 min         1
Name: count, dtype: int64

Some values in `rating` look like durations (e.g. `74 min`). Let's inspect those rows.

In [14]:
df[df["rating"].str.contains("min", na=False)]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,"April 4, 2017",2017,74 min,NaN,Movies,"Louis C.K. muses on religion, eternal love, giving dogs ..."
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,"September 16, 2016",2010,84 min,NaN,Movies,Emmy-winning comedy writer Louis C.K. brings his caustic...
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,"August 15, 2016",2015,66 min,NaN,Movies,The comic puts his trademark hilarious/thought-provoking...


In [15]:
df["release_year"].describe()

count    8807.000000
mean     2014.180198
std         8.819312
min      1925.000000
25%      2013.000000
50%      2017.000000
75%      2019.000000
max      2021.000000
Name: release_year, dtype: float64

## My Observations

### Size and structure
- The dataset has **8,807 rows and 12 columns**.
- `show_id` is unique with no nulls, so it can act as a primary key.
- There are **no duplicate rows** and no duplicate `show_id`s.
- Only `release_year` is numeric. `date_added`, `rating` and `duration` are stored as text and need conversion.

### Content mix
- **Movies are ~70%** of the catalog (6,131) and **TV shows ~30%** (2,676).
- Content skews toward mature audiences: **TV-MA (3,207) and TV-14 (2,160)** together make up roughly 61% of titles.
- Some rating categories overlap or are rare: `NR` (80) and `UR` (3) both mean "unrated", and `TV-Y7-FV`, `NC-17` and `UR` have very few titles.
- Release year: median **2017**, 75% of titles released in **2013 or later**, earliest **1925**. The mean (~2014) is pulled below the median by a small tail of very old titles.
- The latest `date_added` values are from September 2021, so **2021 is likely a partial year** in time-based charts.

### Missing values
| Column | Missing | % |
|---|---|---|
| director | 2,634 | 29.91% |
| country | 831 | 9.44% |
| cast | 825 | 9.37% |
| date_added | 10 | 0.11% |
| rating | 4 | 0.05% |
| duration | 3 | 0.03% |

- `director` has the most missing values. Many TV shows appear to have no director listed, so the gaps may be systematic rather than random.
- All other columns are complete.

### Data-entry problems
- **3 rows have their runtime in the `rating` column** (`66 min`, `74 min`, `84 min`). All are Louis C.K. stand-up specials, and their `duration` is NaN. The values were entered in the wrong column.
- `country`, `cast`, `director` and `listed_in` contain **multiple comma-separated values** in one cell, so they must be split and exploded before counting countries or genres.